# My first MD simulation

672 Lennard-Jones atoms in two dimensions at reduced density ρ* = 0.65, held at the temperature set by `variable T` in the input file (T* = 0.6 to begin with; the exercise asks for 0.45 and 0.3 as well).

Two numbers tell the phases apart:
* the potential energy per atom — low and steady in a solid, higher in a fluid;
* the mean-square displacement ⟨Δr²⟩ (`c_msd[4]`) — flat in a solid, a straight line in a fluid (in 2D its slope is 4D).

Run the cells below after each simulation. The last cell compares all your runs. Exercise: add the pressure (`log.get("Press")`) to the plots — how does it change with temperature, and why?

In [ ]:
# lammps-logfile is served from Atomify's own package index (no network needed).
%pip install -q lammps-logfile pandas

In [ ]:
import glob, os, time
import numpy as np
import lammps_logfile
import matplotlib.pyplot as plt

# Atomify stores every run of this project in runs/<run-name>/ next to this
# notebook (input snapshot, log.lammps, dumps). Pick the newest run here;
# use logs[0], logs[1], ... to look at older ones.
logs = sorted(glob.glob("runs/*/log.lammps"))
if not logs:
    raise RuntimeError("No runs yet: press Run in Atomify, wait for it to finish, then re-run this cell.")
print("Runs found:", *logs, sep="\n  ")

for attempt in range(5):
    try:
        log = lammps_logfile.File(logs[-1])
        break
    except FileNotFoundError:
        # Atomify may still be copying the finished run into the project.
        time.sleep(1)
        os.listdir(os.path.dirname(logs[-1]))
else:
    raise RuntimeError(f"{logs[-1]} is not readable yet: wait for the run to finish, then re-run this cell.")
print("Log keywords:", log.get_keywords())

Temperature, potential energy and mean-square displacement against time for the newest run:

In [ ]:
t   = log.get("Time")
T   = log.get("Temp")
Ep  = log.get("PotEng")       # per atom (LJ units normalise by N)
msd = log.get("c_msd[4]")     # total mean-square displacement, sigma^2

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
axes[0].plot(t, T);   axes[0].set_ylabel("T*");            axes[0].set_title("temperature")
axes[1].plot(t, Ep);  axes[1].set_ylabel("E_pot / N");     axes[1].set_title("potential energy per atom")
axes[2].plot(t, msd); axes[2].set_ylabel("⟨Δr²⟩ / σ²");   axes[2].set_title("mean-square displacement")
for ax in axes: ax.set_xlabel("time t*")
plt.tight_layout(); plt.show()

half = t > t[-1] / 2
slope = np.polyfit(t[half], msd[half], 1)[0]
print(f"mean T* (last half)          = {T[half].mean():.3f}")
print(f"mean E_pot/N (last half)     = {Ep[half].mean():.3f}")
print(f"MSD slope (last half)        = {slope:.3f} sigma^2 per time unit  ->  D = slope/4 = {slope/4:.4f}")

Note down for each temperature:
* What did you predict, and what do you see in the viewer? Gas, liquid, solid — or two phases side by side?
* The potential energy per atom and the MSD slope. Which of the three runs has atoms that do not move away from their neighbours?
* Where are the three runs in the phase diagram of the exercise sheet (the red line is ρ* = 0.65)?

Every run you make in Atomify is kept, so you can compare the three temperatures in one plot:

In [ ]:
for path in logs:
    run = lammps_logfile.File(path)
    plt.plot(run.get("Time"), run.get("c_msd[4]"), label=f'{path.split("/")[1]}: T* = {run.get("Temp").mean():.2f}')
plt.xlabel("time t*"); plt.ylabel("⟨Δr²⟩ / σ²"); plt.legend(); plt.show()